# Lista 13 (8 pkt.)

We wszystkich poniższych zadaniach będziemy używać **OpenSSL** https://www.openssl.org/.

## Krótki tutorial:

### generowanie klucza prywatnego
```
openssl genpkey -algorithm RSA -pkeopt rsa_keygen_bits:2048 -pkeyopt rsa_keygen_pubexp:65537 -out privkey.pem
```
```-algorithm``` wybiera algorytm do generacji klucza, tu ```RSA```

```-pkeyopt rsa_keygen_bits:2048``` ustala długość klucza, tu ```2048``` bitów

```-pkeyopt rsa_keygen_pubexp:65537``` ustala wartość $e$, tu ```65537```

```-out privkey.pem``` zapisuje klucz prywatny w pliku ```privkey.pem```
### generowanie klucza publicznego
```
openssl pkey -in privkey.pem -pubout -out pubkey.pem
```
```-in privkey.pem``` wczytuje plik z kluczem prywatnym

```-pubout``` ustala, że chcemy uzyskać klucz publiczny

```-out pubkey.pem``` zapisuje klucz publiczny w pliku ```pubkey.pem```

### Diffy-Helman
- generowanie publicznych parametrów $p$ i $g$
```
openssl genpkey -genparam -algorithm DH -out dh_params.pem
```
- generowanie klucza prywatnego $a$
```
openssl genpkey -paramfile dh_params.pem -out privkey.pem
```
- generowanie klucza publicznego $g^a$
```
openssl pkey -in privkey.pem -pubout -out pubkey.pem
```
- generowanie wspólnego sekretu $k=(g^b)^a=(g^a)^b$
```
openssl pkeyutl -derive -inkey privkey_A.pem -peerkey pubkey_B.pem -out secret.bin
```

### wyświetlanie w przyjaznej formie
- klucz prywatny
```
openssl pkey -in privkey.pem -text -noout
```
- klucz publiczny
```
openssl pkey -pubin -in pubkey.pem -text -noout
```
- certyfikat
```
openssl x509 -in user.crt -text -noout
```
- parametry Diffiego-Helmana
```
openssl pkeyparam -in dh_params.pem -text -noout
```
### generowanie żądania popisania certyfikatu
```
openssl req -new -key privkey.pem -out req.csr
```
```-key privkey.pem``` wskazuje plik z naszym kluczem prywatnym

```-out req.csr``` zapisuje żądanie certyfikatu do pliku ```req.csr``

po wpisaniu tej komendy pojawi się seria pytań o dane do certyfikatu, można wpisać jakieś rzeczy typu PL, Cracow, UJ itd., można też wpisać '.' (kropkę) i w ten sposób pominąć dane pole

Uwaga: aby certyfikat przechodził później weryfikację bez problemu, dane wpisane w certyfikat użytkownika, i dane wpisane w certyfikat **CA** nie mogą być identyczne.

### generowanie samopodpisanego certyfikatu przez **CA**
```
openssl req -x509 -new -nodes -key rootkey.pem -sha256 -days 1024 -out root.crt
```

```-key rootkey.pem``` wskazuje klucz prywatny **CA**

```-sha256``` wybiera funkcję hashującą używaną do podpisu, tu **SHA 256**

```-days 1024``` ustala termin ważności certyfikatu

```-out root.crt``` zapisuje certyfikat do pliku ```root.crt```

### wygenerowanie i podpisanie certyfiaktu przez **CA** na podstawie żądania
```
openssl x509 -req -in req.csr -CA root.crt -CAkey rootkey.pem -CAcreateserial -out user.crt -days 500 -sha256
```
```-req -in req.csr``` wskazuje żądanie

```-CA root.crt``` wskazuje certyfikat **CA**

```-CAcreateserial``` tworzy numer seryjny cerrtyfikatu

```-out user.crt``` zapisuje podpisany certyfikat do pliku ```user.crt```

```-days 500``` ustala termin ważności certyfikatu, tu ```500``` dni

```-sha256``` wybiera funkcję hashującą używaną do podpisu, tu **SHA 256**
### weryfikacja certyfikatu
```
openssl verify -CAfile root.crt user.crt
```
```-CAfile root.crt``` wskazuje certyfikat **CA** na podstawie którego weryfikujemy certyfiakt użytkownika

```user.crt``` certyfikat, który weryfikujemy
### wydobycie klucza publicznego użytkownika z jego certyfikatu
```
openssl x509 -pubkey -in user.crt -noout > pubkey.pem
```
```-pubkey``` wskazuje, że wyciągamy klucz publiczny

```-in user.crt``` wskazuje plik z certyfikatem

```> pubkey.pem``` zapisuje uzyskany klucz do pliku ```pubkey.pem```

### szyfrowanie pliku za pomocą szyfru asymetrycznego
```
openssl pkeyutl -encrypt -in file.txt -pubin -inkey pubkey.pem -out file.enc
```
```-encrypt``` włącza tryb szyfrowania

```-in file.txt``` wskazuje plik do zaszyfrowania

```-pubin -inkey pubkey.pem``` wskazuje klucz publiczny, którym szyfrujemy

```-out file.enc``` zapsiuje zaszyfrowany plik do pliku ```file.enc```
### deszyfrowanie pliku za pomocą szyfru asymetrycznego
```
openssl pkeyutl -decrypt -in file.enc -inkey privkey.pem -out file.txt
```
```-decrypt``` włącza tryb deszyfrowania

```-in file.enc``` wskazuje plik do deszyfrowania

```-inkey privkey.pem``` wskazuje klucz prywatny, którym deszyfrujemy

```-out fil.txt``` zapsiuje odszyfrowany plik do pliku ```file.txt```

### szyfrowanie pliku za pomocą szyfru symetrycznego
- na podstawie hasła
```
openssl enc -aes-256-cbc -p -pbkdf2 -in file.txt -out file_enc.bin
```
pojawi się zapytanie o hasło, można też dodać ```-pass file:key_pass.pem``` aby pobrać hasło z pliku ```key_pass.pem```

```-aes-256-cbc``` wybiera jako szyfr **AES** z kluczem **256** bitowym w trybie pracy **CBC**

```-p``` wypisuje na ekranie klucz, sól i wektor inicjujący

```--pbkdf2``` ustala metodę używaną przy generowaniu klucza

```-in file.txt``` plik do zaszyfrowania

```file_enc.bin``` zaszyfrowany plik
- na podstawie klucza
```
openssl enc -aes-256-cbc -K XXXXXXXXX -iv XXXXXXXX -p -in file.txt -out file_enc.bin
```
```-aes-256-cbc``` wybiera jako szyfr **AES** z kluczem **256** bitowym w trybie pracy **CBC**

```-K XXXXXXXXX``` ustala klucz na ```XXXXXXXXX```, format szesnastkowy, odpowiednia długość do szyfru

```-iv XXXXXXXXX``` ustala wektor inicjujący na ```XXXXXXXXX```, format szesnastkowy, odpowiednia długość do szyfru

```-p``` wypisuje na ekranie klucz, sól i wektor inicjujący

```-in file.txt``` plik do zaszyfrowania

```file_enc.bin``` zaszyfrowany plik

### deszyfrowanie pliku za pomocą szyfru symetrycznego
- na podstawie hasła
```
openssl enc -aes-256-cbc -d -p -pbkdf2 -in file_enc.bin -out file.txt
```
pojawi się zapytanie o hasło, można też dodać ```-pass file:key_pass.pem``` aby pobrać hasło z pliku ```key_pass.pem```

```-aes-256-cbc``` wybiera jako szyfr **AES** z kluczem **256** bitowym w trybie pracy **CBC**

```-d``` wybiera tryb deszyfracji

```-pass file:key_pass.pem``` wybiera deszyfrowanie na podstawie klucza wygenerowanego z hasła znajdującego się w pliku ```key_pass.pem```

```-p``` wypisuje na ekranie klucz, sól i wektor inicjujący

```-pbkdf2``` ustala funkcję hashującą używaną przy generowaniu klucza

```-in file_enc.bin``` plik do deszyfrowania

```file.txt``` odszyfrowany plik
- na podstawie klucza
```
openssl enc -aes-256-cbc -d -K XXXXXXXXX -iv XXXXXXXX -p -in file.enc -out file.txt
```
```-aes-256-cbc``` wybiera jako szyfr **AES** z kluczem **256** bitowym w trybie pracy **CBC**

```-d``` wybiera tryb deszyfracji

```-K XXXXXXXXX``` ustala klucz na ```XXXXXXXXX```, format szesnastkowy, odpowiednia długość do szyfru

```-iv XXXXXXXXX``` ustala wektor inicjujący na ```XXXXXXXXX```, format szesnastkowy, odpowiednia długość do szyfru

```-p``` wypisuje na ekranie klucz, sól i wektor inicjujący

```-in file.enc``` plik do odszyfrowania

```file.txt``` odszyfrowany plik

### generowanie podpisu elektronicznego
```
openssl dgst -sha1 -sign privkey.pem -out signature.bin file.txt
```
```-sha1``` wybiera funkcję hashującą **SHA 1**

```-sign privkey.pem``` podpisuje kluczem prywatnym ```privkey.pem```

```-out signature.bin``` zapisuje podpis do pliku binarnego ```signature.bin```

```file.txt``` podpisywany plik
### weryfikacja podpisu elektronicznego
```
openssl dgst -sha1 -verify pubkey.pem -signature signature.bin file.txt
```
```-sha1``` wybiera funkcję hashującą **SHA 1**

```-verify pubkey.pem``` weryfikuje podpis kluczem publicznm ```pubkey.pem```

```-signature signature.bin``` wskazuje plik z podpisem do weryfikacji

```file.txt``` plik, którego podpis weryfikujemy

### generowanie pseudo-losowych bajtów
```
openssl rand -base64 32 -out symkey.pem
```
- ```-base64``` ustala kodowanie na base64

- ```32``` liczba bajtów do wygenerowania

- ```-out symkey.pem``` zapisuje ciąg bajtów do symkey.pem


## Zadanie 1 (1 pkt.)

Zdeszyfruj plik **AESencryptedCBC.enc** za pomocą klucza:

8541F781259ADA8631F18A9A8E97771BFE395CF7ACBE5F77

oraz wektora inicjującego:

99B54E58FF44F7599465E6888A3C6C1E

w trybie **CBC** z kluczem **192** bitowym.

In [14]:
!openssl enc -d -aes-192-cbc -in AESencrypted.enc -out zadanie1_jawny.txt -K 8541F781259ADA8631F18A9A8E97771BFE395CF7ACBE5F77 -iv 99B54E58FF44F7599465E6888A3C6C1E

## Zadanie 2 (1 pkt.)

Zaszyfruj plik **kamien.txt** za pomocą hasła w trybie **OFB** z kluczem **256** bitowym.

In [15]:
!openssl enc -aes-256-ofb -pbkdf2 -in kamien.txt -out kamien.enc -pass pass:student

## Zadanie 3 (1 pkt.)

Stwórz dwa foldery, **Alice** oraz **Bob**, i przeprowadź protokół uzgadniania klucza za pomocą metody Diffiego-Helmana pomiędzy tymi folderami. Gdy uzsykasz w obu folderach plik **secret.bin**, który jest plikiem binarnym zawierającym uzgodniony ciąg bitów pomiędzy Alicją i Bobem, zaszyfruj plik **rozwijajac_rilkego.txt** w folderze Alicji podając jako hasło plik **secret.bin** a następnie zdeszyfruj go w folderze Boba za pomocą jego pliku **secret.bin**. Użyj AES 256 w dowolnym trybie.

In [16]:
%%bash
rm -rf zad3_Alice zad3_Bob
mkdir -p zad3_Alice zad3_Bob

# 1. Generowanie parametrów DH
openssl genpkey -genparam -algorithm DH -out dh_params.pem

# 2. Generowanie kluczy dla Alice
openssl genpkey -paramfile dh_params.pem -out zad3_Alice/privkey.pem
openssl pkey -in zad3_Alice/privkey.pem -pubout -out zad3_Alice/pubkey.pem

# 3. Generowanie kluczy dla Boba
openssl genpkey -paramfile dh_params.pem -out zad3_Bob/privkey.pem
openssl pkey -in zad3_Bob/privkey.pem -pubout -out zad3_Bob/pubkey.pem

# 4. Wymiana kluczy i generowanie sekretu (Alice)
openssl pkeyutl -derive -inkey zad3_Alice/privkey.pem -peerkey zad3_Bob/pubkey.pem -out zad3_Alice/secret.bin

# 5. Wymiana kluczy i generowanie sekretu (Bob)
openssl pkeyutl -derive -inkey zad3_Bob/privkey.pem -peerkey zad3_Alice/pubkey.pem -out zad3_Bob/secret.bin

# 6. Szyfrowanie przez Alice
openssl enc -aes-256-cbc -p -pbkdf2 -in rozwijajac_rilkego.txt -out zad3_Alice/message.enc -pass file:zad3_Alice/secret.bin

# Symulacja przesłania pliku
cp zad3_Alice/message.enc zad3_Bob/message.enc

# 7. Deszyfracja przez Boba
openssl enc -aes-256-cbc -d -p -pbkdf2 -in zad3_Bob/message.enc -out zad3_Bob/rozwijajac_rilkego_decrypted.txt -pass file:zad3_Bob/secret.bin

..

...................................................................................................................................................................................................+...................................................................................................+.............+.............................................................................................................+......+....................................................................+..........+.........................................................+.............+......................................................................................................................................................................................................................................................................................................+...................+.................................................+........................................................

salt=370550A8BA64CB79
key=31065E1B0F085E652031B09FEE64CF45EA314D9E18EB553E39E75695817C8F21
iv =CE087EECF8E311426C321E35B1A4BD71
salt=370550A8BA64CB79
key=31065E1B0F085E652031B09FEE64CF45EA314D9E18EB553E39E75695817C8F21
iv =CE087EECF8E311426C321E35B1A4BD71


## Zadanie 4 (1 pkt.)

Klikając w kłodkę w przeglądarce internetowej otwórz certyfikaty następujących stron:

- Wikipedia
- Google
- Strona tego kursu kryptografii

Odczytaj z tych certyfikatów następujące informacje:
- data ważności certyfikatu
- rodzaj używanego algorytmu asymetrycznego
- klucz publiczny
- rodzaj używanego algorytmu do podpisu

# Wikipedia
Validity
Not Before
Mon, 08 Dec 2025 22:27:50 GMT
Not After
Sun, 08 Mar 2026 22:27:49 GMT 

Elliptic Curve

04:D3:DC:38:AE:EA:0E:52:38:9B:C3:74:7F:DE:36:0B:0F:94:42:03:51:B4:7E:74:77:51:D0:94:1B:CD:17:8D:D0:D8:44:7A:D4:E0:58:AB:BE:DB:F7:B0:11:A5:90:08:2A:14:73:7F:7A:87:06:38:0A:E3:B6:A1:AA:60:3C:E9:8E

ECDSA with SHA-384

# Google
Validity
Not Before
Tue, 09 Dec 2025 17:08:50 GMT
Not After
Tue, 03 Mar 2026 17:08:49 GMT 

Elliptic Curve

04:C5:F5:DF:F2:FA:FB:47:56:BD:8F:A7:EB:61:0C:8E:0A:9A:65:0B:C4:93:BE:FD:0B:0E:DF:F7:CC:38:CC:54:1A:11:73:01:14:1C:4A:8D:9A:D0:93:65:E4:E2:E3:58:FA:08:29:E2:C4:7E:3B:1F:47:56:ED:FA:D0:04:71:2C:6A

SHA-256 with RSA Encryption

# Kurs
Validity
Not Before
Thu, 27 Nov 2025 13:23:26 GMT
Not After
Fri, 27 Nov 2026 13:23:26 GMT 

RSA

A2:40:C5:DC:98:94:A6:50:FA:65:24:CE:16:09:B5:FD:72:AC:46:2C:80:65:B0:92:FE:87:AB:15:FE:DB:05:24:F0:99:28:33:10:71:57:B4:4B:1F:A0:2F:41:39:EF:75:AC:AC:00:12:3F:EB:4D:05:AB:A6:9E:E2:B9:57:03:FB:F7:EF:EB:90:7B:40:97:1D:F8:DC:2E:5E:66:3D:DD:52:D3:E0:0A:88:95:52:09:93:6F:0E:31:1C:62:65:C5:98:3B:0E:29:55:22:D2:64:9D:68:B1:4C:C5:3C:E3:74:B0:51:52:A1:C3:6C:8C:AC:60:50:DD:83:62:30:D2:B5:26:CC:A9:D6:6C:ED:BB:9E:4A:D0:B1:EB:ED:11:CA:38:E0:A8:78:D6:47:E0:0B:B1:3B:BE:05:8A:CF:7C:F7:97:06:DF:D1:14:6F:73:A0:AC:9A:0A:2F:E1:AC:04:DE:78:D8:3D:A9:61:A7:66:29:47:9B:2F:60:D2:72:F9:9C:46:FF:18:CF:4C:85:A1:5B:18:C3:51:AC:B5:9F:B5:5E:7E:2A:09:B2:E6:7A:52:DA:53:16:E1:7D:D0:A5:CD:20:6E:D8:8A:38:DF:71:D7:37:A0:94:95:27:0D:F0:84:0D:3F:78:9E:9A:56:38:43:FA:9C:94:04:64:45:13:DE:0C:30:EF:2F:CA:DD:6D:EC:94:A0:83:52:0B:E9:EB:70:2F:FF:B7:6A:36:DE:BB:DB:8B:28:7A:46:70:7C:1A:7D:D8:A4:3B:4E:40:F6:FF:FD:A9:D9:3F:B1:01:3E:D0:8B:3E:E0:37:50:35:88:04:D9:52:24:00:EC:F1:2B:42:B3:4E:DA:FF:61:75:0E:9C:24:76:85:CD:AA:44:B8:21:E1:39:6D:DE:4D:C3:74:3F:FC:D4:61:9E:6B:6C:4E:C5:DC:A1:4B:4A:A0:94:5D:AF:FA:88:44:19:5F:51:B4:CD:39:2D:A6:52:7D:69:3F:55:0A:FF:76:41:41:99:79:88:8F:2C:84:06:0C:96:B1:DA:43:07:30:9B:51:C2:1D:65:A4:E9:4D:F3:56:AF:5A:81:5D:AE:C0:AF:D8:6A:36:CE:20:3C:29:0B:C6:E2:73:DC:EB:93:A6:D5:17:30:E2:FE:F1:E8:DF:0D:AC:27:B1:4F:08:AD:F2:FA:43:90:AA:12:D7:D8:BD:97:F8:C5:E2:60:86:D2:69:A2:48:47:73:70:C4:98:C4:BB:0B:5E:64:D8:9C:57:86:B3:A4:7D:CA:53:31:C4:CA:45:E3:C4:E2:7D:EE:81:AF:32:20:61:EA:72:A2:A3:E8:31:48:F3:64:78:7B:6B:46:47:F8:46:2F:71:53:91:41:A5

SHA-256 with RSA Encryption

## Zadanie 5 (1 pkt.)

Spróbuj zaszyfrować plik bajka.txt za pomocą RSA. Co się dzieje?

In [17]:
!openssl genpkey -algorithm RSA -out rsa_test_priv.pem -pkeyopt rsa_keygen_bits:2048
!openssl pkey -in rsa_test_priv.pem -pubout -out rsa_test_pub.pem

!openssl pkeyutl -encrypt -in bajka.txt -pubin -inkey rsa_test_pub.pem -out bajka.enc

......+.........+..+...+.+..+...+....+...........+.+...+...+..+++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++*..+............+.....+.......+..+....+......+...+......+..+++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++*.....+.........................+............+...+........+...................+...+..+...+....+.....+....+.....+....+..+...+...+...............+....+......+..+.+...........+..........+..................+..+......+...+......+....+............+..+.+...............+.........+...............+................................+...+...+...+......................+...+..+....+.....+..........+...+......+.....+....+...........+.......+........+.+........+..........+......+.....+....+..+.............+..+......+............+...+..........+...+......+......+..+..........+..+.............+..................+...+......+...+........+...+.+.........+.........+..+...+.+.....+............+...+....+...+..+......+.+.....+.........+....+........+....+.....+.....

## Zadanie 6 (1 pkt.)

Sprawdź, który z certyfikatów **ToJa1.crt**, **ToJa2.crt** czy **ToJa3.crt** jest prawdziwy. Certyfikat zaufanej trzeciej strony to **root.crt**.

Następnie wyekstrahuj klucz publiczny z prawdziwego certyfikatu i sprawdź, który z plików **plik1.txt**, **plik2.txt** czy **plik3.txt**, zostały podpisane przez właściciela certyfikatu. Podpisy plików znajdują się odpowiednio w plikach **signature1.bin**, **signature2.bin** i **signature3.bin**. Pliki podpisano za pomocą **SHA 1**.

In [18]:
print("--- Sprawdzanie certyfikatu 1 ---")
!openssl verify -CAfile root.crt ToJa1.crt
print("--- Sprawdzanie certyfikatu 2 ---")
!openssl verify -CAfile root.crt ToJa2.crt
print("--- Sprawdzanie certyfikatu 3 ---")
!openssl verify -CAfile root.crt ToJa3.crt

--- Sprawdzanie certyfikatu 1 ---
C = Pl, ST = KRK, O = UJ
error 7 at 0 depth lookup: certificate signature failure
error ToJa1.crt: verification failed
402752F5837F0000:error:02000068:rsa routines:ossl_rsa_verify:bad signature:../crypto/rsa/rsa_sign.c:430:
402752F5837F0000:error:1C880004:Provider routines:rsa_verify:RSA lib:../providers/implementations/signature/rsa_sig.c:774:
402752F5837F0000:error:06880006:asn1 encoding routines:ASN1_item_verify_ctx:EVP lib:../crypto/asn1/a_verify.c:217:
--- Sprawdzanie certyfikatu 2 ---
ToJa2.crt: OK
--- Sprawdzanie certyfikatu 3 ---
C = PL, ST = KRK, O = Internet Widgits Pty Ltd
error 7 at 0 depth lookup: certificate signature failure
error ToJa3.crt: verification failed
40C78F2EB67F0000:error:02000068:rsa routines:ossl_rsa_verify:bad signature:../crypto/rsa/rsa_sign.c:430:
40C78F2EB67F0000:error:1C880004:Provider routines:rsa_verify:RSA lib:../providers/implementations/signature/rsa_sig.c:774:
40C78F2EB67F0000:error:06880006:asn1 encoding routine

In [19]:
!openssl x509 -pubkey -in ToJa2.crt -noout > extracted_pubkey.pem

print("Weryfikacja plik1:")
!openssl dgst -sha1 -verify extracted_pubkey.pem -signature signature1.bin plik1.txt
print("Weryfikacja plik2:")
!openssl dgst -sha1 -verify extracted_pubkey.pem -signature signature2.bin plik2.txt
print("Weryfikacja plik3:")
!openssl dgst -sha1 -verify extracted_pubkey.pem -signature signature3.bin plik3.txt

Weryfikacja plik1:
Verified OK
Weryfikacja plik2:
Verification failure
40577335E77F0000:error:0200008A:rsa routines:RSA_padding_check_PKCS1_type_1:invalid padding:../crypto/rsa/rsa_pk1.c:79:
40577335E77F0000:error:02000072:rsa routines:rsa_ossl_public_decrypt:padding check failed:../crypto/rsa/rsa_ossl.c:697:
40577335E77F0000:error:1C880004:Provider routines:rsa_verify:RSA lib:../providers/implementations/signature/rsa_sig.c:774:
Weryfikacja plik3:
Verification failure
4067DCF56D7F0000:error:0200008A:rsa routines:RSA_padding_check_PKCS1_type_1:invalid padding:../crypto/rsa/rsa_pk1.c:79:
4067DCF56D7F0000:error:02000072:rsa routines:rsa_ossl_public_decrypt:padding check failed:../crypto/rsa/rsa_ossl.c:697:
4067DCF56D7F0000:error:1C880004:Provider routines:rsa_verify:RSA lib:../providers/implementations/signature/rsa_sig.c:774:


## Zadanie 7 (2 pkt.)

W tym zadaniu przećwiczymy bardziej pełny schemat komunikacji. Stwórz trzy foldery **Alice**, **Bob**, **CA**. Alicja będzie nadawcą pliku, Bob odbiorcą a CA to zaufana trzecia strona.

Użyj następujących algorytmów:
- RSA 2048 e=65537
- SHA 256
- AES 256 CBC z kluczem generowanym z hasła

1. Każda ze stron generuje swoje pary klucz prywatny **privkey_Alice.pem** i klucz publiczny **pubkey_Alice.pem**.
2. W ogólości klucz prywatny należy zaszyfrować jakimś szyfrem symetrycznym, np. **AES** aby go bezpiecznie przechowywać, tu dla ułatwienia pomieniemy ten krok.
3. Zaufana trzecia strona **CA** tworzy swój certyfikat, podpisany przez samą siebie **root.crt**, który jest ogłaszany publicznie, tzn. trafia do Alicji i Boba.
4. Następnie Alicja i Bob muszą uzyskać certyfikaty podpisane przez **CA**. W związku z tym każde z nich tworzy rządanie certyfikatu **Alice_req.csr**.
5. Alicja i Bob wysyłają swoje żądania do **CA**, które po zweryfikowaniu Alicji i Boba podpisuje je. Podpisane certyfikaty Alicji i Boba sa ogłaszane publicznie.
6. Alicja bierze certyfikat Boba weryfikuje go za pomocą certyfikatu **CA** (**root.crt**). Jeśli certyfikat jest poprawny, wyciąga z niego klucz publiczny Boba i zapisuje do pliku **pubkey_Bob.pem**.
7. Alicja generuje losowe hasło (**sympass.pem**) z którego będzie generowany klucz symetryczny, które będzie chciała przekazać Bobowi.
8. Alicja szyfruje hasło **sympass.pem** za pomocą klucza publicznego Boba **pubkey_Bob.pem** otrzymując **sympass.enc**.
7. Alicja podpisuje zaszyfrowane hasło poprzez zhashowanie go i zaszyfrowanie za pomocą swojego klucza prywatnego **privkey_ALice.pem**. Podpis zapisuje w pliku **signature.bin**.
8. Alicja wysyła do Boba zaszyfrowane hasło **sympass.enc** oraz jego podpis **signature.bin**.
9. Bob weryfikuje certyfikat Alicji, wyciąga z niego jej klucz publiczny, deszyfruje otrzymane hasło **sympass.enc** i sprawdza podpis.
10. Alicja szyfruje plik, który chce przekazać Bobowi za pomocą hasła **sympass.pem** i wysyła zaszyfrowany plik do Boba.
11. Bob deszyfruje otrzymany plik za pomocą **sympass.pem**.

In [20]:
%%bash
rm -rf Alice Bob CA
mkdir Alice Bob CA

# === KROK 1 i 2 ===
# Alice
openssl genpkey -algorithm RSA -pkeyopt rsa_keygen_bits:2048 -out Alice/privkey_Alice.pem
openssl pkey -in Alice/privkey_Alice.pem -pubout -out Alice/pubkey_Alice.pem
# Bob
openssl genpkey -algorithm RSA -pkeyopt rsa_keygen_bits:2048 -out Bob/privkey_Bob.pem
openssl pkey -in Bob/privkey_Bob.pem -pubout -out Bob/pubkey_Bob.pem
# CA (Root)
openssl genpkey -algorithm RSA -pkeyopt rsa_keygen_bits:2048 -out CA/rootkey.pem

# === KROK 3 ===
openssl req -x509 -new -nodes -key CA/rootkey.pem -sha256 -days 1024 -out CA/root.crt -subj "/C=PL/ST=Malopolska/L=Krakow/O=RootCA/CN=RootCA"
cp CA/root.crt Alice/root.crt
cp CA/root.crt Bob/root.crt

# === KROK 4 i 5 ===
openssl req -new -key Alice/privkey_Alice.pem -out Alice/Alice_req.csr -subj "/C=PL/ST=Malopolska/L=Krakow/O=AliceInc/CN=Alice"
openssl req -new -key Bob/privkey_Bob.pem -out Bob/Bob_req.csr -subj "/C=PL/ST=Malopolska/L=Krakow/O=BobCorp/CN=Bob"

openssl x509 -req -in Alice/Alice_req.csr -CA CA/root.crt -CAkey CA/rootkey.pem -CAcreateserial -out Alice/Alice.crt -days 500 -sha256
openssl x509 -req -in Bob/Bob_req.csr -CA CA/root.crt -CAkey CA/rootkey.pem -CAcreateserial -out Bob/Bob.crt -days 500 -sha256

cp Bob/Bob.crt Alice/Bob.crt
cp Alice/Alice.crt Bob/Alice.crt

# === KROK 6 ===
openssl verify -CAfile Alice/root.crt Alice/Bob.crt

openssl x509 -pubkey -in Alice/Bob.crt -noout > Alice/pubkey_Bob_extracted.pem

# === KROK 7 ===
openssl rand -base64 32 > Alice/sympass.pem

# === KROK 8 ===
openssl pkeyutl -encrypt -in Alice/sympass.pem -pubin -inkey Alice/pubkey_Bob_extracted.pem -out Alice/sympass.enc

# === KROK 9 ===
openssl dgst -sha256 -sign Alice/privkey_Alice.pem -out Alice/signature.bin Alice/sympass.enc

# === KROK 10 ===
cp Alice/sympass.enc Bob/sympass.enc
cp Alice/signature.bin Bob/signature.bin

# === KROK 11 ===
openssl x509 -pubkey -in Bob/Alice.crt -noout > Bob/pubkey_Alice_extracted.pem

openssl dgst -sha256 -verify Bob/pubkey_Alice_extracted.pem -signature Bob/signature.bin Bob/sympass.enc

openssl pkeyutl -decrypt -in Bob/sympass.enc -inkey Bob/privkey_Bob.pem -out Bob/sympass_decrypted.pem

# === KROK 12 i 13 ===
# Alicja
echo "Super ekstra tajny plik dla Boba" > Alice/tajne_dane.txt
openssl enc -aes-256-cbc -pbkdf2 -in Alice/tajne_dane.txt -out Alice/tajne_dane.enc -pass file:Alice/sympass.pem

# Przesłanie
cp Alice/tajne_dane.enc Bob/tajne_dane.enc

# Bob
openssl enc -d -aes-256-cbc -pbkdf2 -in Bob/tajne_dane.enc -out Bob/tajne_dane_decrypted.txt -pass file:Bob/sympass_decrypted.pem

# Sprawdzenie
cat Bob/tajne_dane_decrypted.txt

....+.....+....+..+...+...+.............+......+...++++++++++++++++++++++++++++++++++++++++

+++++++++++++++++++++++++*.........+......+.............+.........+..+.+........+..........+..+++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++*..+..+...+...+....+.....+......+............+.+......+.....+......+.............+.....+.+........+.............+.......................+.+.....+.+............+..+...+...+.........+.........+.+...+...+........+.........+...............+...+.........+.+........+.......+.....+...+....+..+.+..............................+...+......+++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
........+.......+........+...+............+......+.+..................+...+..+..........+.....+.+.....+....+...+...+..+...+......+.+..+...............+.......+.....+.+..+.......+.....+++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++*.....+.......+..+.+...........+......+...+.+.....+....+............+...+..+....+++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++*....+...+...........+.+...+...+..+.+.....++++++++++

Alice/Bob.crt: OK
Verified OK
Super ekstra tajny plik dla Boba
